# Oracle on Azure Workshop - Admin Notebook

Dieses Notebook ermöglicht:
- SSH-Verbindung zur User00 VM via Service Principal
- Verbindung zur Oracle Autonomous Database (ADB) im ODAA VNet
- Netzwerk-Diagnose und Tests

## Helper Functions & Configuration

In [1]:
# Helper functions for Oracle on Azure Workshop
import subprocess
import sys
import json
import os
import re

# Determine directories relative to notebook location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
TERRAFORM_DIR = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == "notebook" else NOTEBOOK_DIR
if not os.path.exists(os.path.join(TERRAFORM_DIR, "terraform.tfvars")):
    # Fallback: look in parent
    TERRAFORM_DIR = os.path.dirname(TERRAFORM_DIR)
    
print(f"Terraform dir: {TERRAFORM_DIR}")

def run(cmd, shell=True, cwd=None, timeout=None):
    """Run command and print output. Returns exit code."""
    result = subprocess.run(
        cmd,
        shell=shell,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result.returncode

def run_output(cmd, shell=True, cwd=None, timeout=None):
    """Run command and return stdout as string."""
    result = subprocess.run(
        cmd,
        shell=shell,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    return result.stdout.strip() if result.stdout else ""

def run_args(args, cwd=None, timeout=None):
    """Run a command without shell."""
    result = subprocess.run(
        args,
        shell=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result.returncode

def az_json(cmd):
    """Run az CLI command and return parsed JSON."""
    raw = run_output(cmd)
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None

def parse_tfvars(path):
    """Parse terraform.tfvars file."""
    values = {}
    if not os.path.exists(path):
        return values
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("//"):
                continue
            line = re.split(r"\s+#", line, maxsplit=1)[0].strip()
            m = re.match(r"^(?P<key>[A-Za-z0-9_]+)\s*=\s*(?P<val>.+)$", line)
            if not m:
                continue
            key = m.group("key")
            val = m.group("val").strip()
            if val.startswith('"') and val.endswith('"'):
                values[key] = val[1:-1]
    return values

print("✅ Helper functions loaded.")
print(f"   Terraform dir: {TERRAFORM_DIR}")

Terraform dir: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform
✅ Helper functions loaded.
   Terraform dir: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform


## Service Principal Login

Authentifizierung mit dem Service Principal aus `terraform.tfvars`.
Der SP hat bereits `Virtual Machine Administrator Login` Rechte auf Subscription-Ebene.

In [2]:
# ============================================================================
# Service Principal Authentication (from terraform.tfvars)
# ============================================================================

TFVARS_PATH = os.path.join(TERRAFORM_DIR, "terraform.tfvars")
NOTEBOOK_AZURE_CONFIG = os.path.join(TERRAFORM_DIR, ".azure-notebook")
os.makedirs(NOTEBOOK_AZURE_CONFIG, exist_ok=True)
os.environ["AZURE_CONFIG_DIR"] = NOTEBOOK_AZURE_CONFIG

tfvars = parse_tfvars(TFVARS_PATH)

# Check required variables (matching terraform.tfvars variable names)
required = ["client_id", "client_secret", "tenant_id", "vm_subscription_id"]
missing = [k for k in required if not tfvars.get(k)]
if missing:
    raise ValueError(f"Fehlende Felder in terraform.tfvars: {', '.join(missing)}")

CLIENT_ID = tfvars["client_id"]
CLIENT_SECRET = tfvars["client_secret"]
TENANT_ID = tfvars["tenant_id"]
VM_SUBSCRIPTION_ID = tfvars["vm_subscription_id"]
ODAA_SUBSCRIPTION_ID = tfvars.get("odaa_subscription_id", "")
LOCATION = tfvars.get("location", "francecentral")

print(f"✅ Konfiguration geladen")
print(f"   Tenant ID: {TENANT_ID}")
print(f"   VM Subscription: {VM_SUBSCRIPTION_ID}")
print(f"   ODAA Subscription: {ODAA_SUBSCRIPTION_ID}")
print(f"   Location: {LOCATION}")
print(f"   Isolierter Azure CLI Kontext: {NOTEBOOK_AZURE_CONFIG}")

def az_cmd(args):
    """Run Azure CLI via cmd.exe (works with az.cmd shims on Windows)."""
    return run_args(["cmd", "/c"] + args)

# Login with Service Principal
rc = az_cmd([
    "az", "login", "--service-principal",
    "-u", CLIENT_ID,
    f"--password={CLIENT_SECRET}",
    "--tenant", TENANT_ID,
    "--only-show-errors", "--output", "none",
])
if rc != 0:
    raise SystemExit("❌ SP Login fehlgeschlagen")

# Set subscription to VM subscription
rc = az_cmd(["az", "account", "set", "-s", VM_SUBSCRIPTION_ID, "--only-show-errors", "--output", "none"])
if rc != 0:
    raise SystemExit("❌ az account set fehlgeschlagen")

current_account = run_output("az account show --query user.name -o tsv --only-show-errors")
print(f"✅ SP Login erfolgreich: {current_account}")

✅ Konfiguration geladen
   Tenant ID: f71980b2-590a-4de9-90d5-6fbc867da951
   VM Subscription: 556f9b63-ebc9-4c7e-8437-9a05aa8cdb25
   ODAA Subscription: 4aecf0e8-2fe2-4187-bc93-0356bd2676f5
   Location: francecentral
   Isolierter Azure CLI Kontext: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform\.azure-notebook
✅ SP Login erfolgreich: 8a9f736e-4eb2-4484-ae90-2493f57102b3


## Manage Users (Password Rotation + MFA Reset)

Kombiniertes Script `manage-users.ps1` für Workshop-User-Verwaltung:
- **rotate-passwords**: Neue Passwörter generieren via `az ad user update` (kein Terraform nötig)
- **reset-mfa**: Alle MFA-Methoden entfernen (Authenticator, Phone, etc.)
- **reset-all**: Beides in einem Durchlauf

Die Credentials werden direkt in `user_credentials.json` aktualisiert.

In [3]:
import os, json, subprocess, secrets, string, datetime, urllib.parse

# Action: "rotate-passwords" | "reset-mfa" | "reset-all"
ACTION = "reset-all"

# Optional metadata
EVENT_NAME = "workshop-march-2026"

# Safety switches
WHAT_IF = False
FORCE_CHANGE_ON_LOGIN = True
PASSWORD_LENGTH = 16

script_path = os.path.join(TERRAFORM_DIR, "scripts", "manage-users.ps1")
cred_file = os.path.join(TERRAFORM_DIR, "user_credentials.json")
if not os.path.exists(cred_file):
    cred_file = os.path.join(TERRAFORM_DIR, "identity", "user_credentials.json")

if ACTION not in ("rotate-passwords", "reset-mfa", "reset-all"):
    raise ValueError("ACTION must be rotate-passwords, reset-mfa, or reset-all")

if not os.path.exists(cred_file):
    raise FileNotFoundError(f"Credentials file not found: {cred_file}")

# Ensure notebook az context is reused (set in SP login cell)
os.environ["AZURE_CONFIG_DIR"] = NOTEBOOK_AZURE_CONFIG

def az_cli(args):
    """Run az via cmd shim and return (rc, stdout, stderr)."""
    p = subprocess.run(
        ["cmd", "/c", "az"] + args,
        shell=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=TERRAFORM_DIR,
    )
    return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()

def new_secure_password(length=16):
    upper = "ABCDEFGHJKLMNPQRSTUVWXYZ"
    lower = "abcdefghjkmnpqrstuvwxyz"
    digits = "23456789"
    special = "@#*+-="

    # Ensure at least one char from each category
    chars = [
        secrets.choice(upper),
        secrets.choice(lower),
        secrets.choice(digits),
        secrets.choice(special),
    ]

    all_chars = upper + lower + digits + special
    while len(chars) < length:
        chars.append(secrets.choice(all_chars))

    secrets.SystemRandom().shuffle(chars)
    return "".join(chars)

def mfa_delete_uri(upn, method_type, method_id):
    upn_q = urllib.parse.quote(upn, safe="")
    base = f"https://graph.microsoft.com/v1.0/users/{upn_q}/authentication"
    mapping = {
        "phoneAuthenticationMethod": "phoneMethods",
        "microsoftAuthenticatorAuthenticationMethod": "microsoftAuthenticatorMethods",
        "softwareOathAuthenticationMethod": "softwareOathMethods",
        "fido2AuthenticationMethod": "fido2Methods",
        "windowsHelloForBusinessAuthenticationMethod": "windowsHelloForBusinessMethods",
        "emailAuthenticationMethod": "emailMethods",
        "temporaryAccessPassAuthenticationMethod": "temporaryAccessPassMethods",
    }
    segment = mapping.get(method_type)
    return f"{base}/{segment}/{method_id}" if segment else None

with open(cred_file, "r", encoding="utf-8") as f:
    credentials = json.load(f)

users = credentials.get("users", {})
if not users:
    raise ValueError(f"No users found in {cred_file}")

do_passwords = ACTION in ("rotate-passwords", "reset-all")
do_mfa = ACTION in ("reset-mfa", "reset-all")

print("=" * 72)
print("MANAGE USERS (Notebook / SP context)")
print("=" * 72)
print(f"Action: {ACTION}")
print(f"Users: {len(users)}")
print(f"Credentials file: {cred_file}")
print(f"WhatIf: {WHAT_IF}")
print("=" * 72)

# Validate active az login context
rc, out, err = az_cli(["account", "show", "--query", "user.name", "-o", "tsv", "--only-show-errors"])
if rc != 0:
    raise SystemExit(f"az account show failed: {err or out}")
print(f"Active AZ identity: {out}")

stats = {
    "password_success": 0,
    "password_error": 0,
    "mfa_success": 0,
    "mfa_error": 0,
}

for user_key, user_data in users.items():
    upn = user_data.get("user_principal_name")
    print(f"\n[{user_key}] {upn}")

    # Password rotation via Azure CLI
    if do_passwords:
        pwd = new_secure_password(PASSWORD_LENGTH)
        if WHAT_IF:
            print(f"  [WhatIf] Would set new password ({PASSWORD_LENGTH} chars)")
            stats["password_success"] += 1
        else:
            rc, out, err = az_cli([
                "ad", "user", "update",
                "--id", upn,
                "--password", pwd,
                "--force-change-password-next-sign-in", str(FORCE_CHANGE_ON_LOGIN).lower(),
                "--only-show-errors",
                "--output", "none",
            ])
            if rc == 0:
                print("  Password rotated")
                user_data["pw"] = pwd
                user_data["password"] = pwd
                stats["password_success"] += 1
            else:
                print(f"  ERROR password update: {err or out}")
                stats["password_error"] += 1

    # MFA reset via Graph endpoint using az rest (SP token)
    if do_mfa:
        upn_q = urllib.parse.quote(upn, safe="")
        rc, out, err = az_cli([
            "rest",
            "--method", "GET",
            "--uri", f"https://graph.microsoft.com/v1.0/users/{upn_q}/authentication/methods",
            "--output", "json",
            "--only-show-errors",
        ])
        if rc != 0:
            print(f"  ERROR get MFA methods: {err or out}")
            stats["mfa_error"] += 1
        else:
            methods = json.loads(out).get("value", [])
            mfa_methods = [m for m in methods if m.get("@odata.type") != "#microsoft.graph.passwordAuthenticationMethod"]
            if not mfa_methods:
                print("  No MFA methods registered")
                stats["mfa_success"] += 1
            else:
                print(f"  Found {len(mfa_methods)} MFA method(s)")
                ok = True
                for m in mfa_methods:
                    method_type = (m.get("@odata.type") or "").replace("#microsoft.graph.", "")
                    method_id = m.get("id")
                    del_uri = mfa_delete_uri(upn, method_type, method_id)
                    if not del_uri:
                        print(f"  Skip unsupported method: {method_type}")
                        continue

                    if WHAT_IF:
                        print(f"  [WhatIf] Would remove: {method_type}")
                        continue

                    rc_del, out_del, err_del = az_cli([
                        "rest",
                        "--method", "DELETE",
                        "--uri", del_uri,
                        "--only-show-errors",
                        "--output", "none",
                    ])
                    if rc_del == 0:
                        print(f"  Removed: {method_type}")
                    else:
                        ok = False
                        print(f"  ERROR remove {method_type}: {err_del or out_del}")

                if ok:
                    stats["mfa_success"] += 1
                else:
                    stats["mfa_error"] += 1

# Update credentials metadata
if not WHAT_IF and do_passwords and stats["password_success"] > 0:
    credentials["generated_at"] = datetime.datetime.now(datetime.timezone.utc).astimezone().isoformat()
    if EVENT_NAME:
        credentials["last_event"] = EVENT_NAME
    credentials["last_action"] = ACTION

    with open(cred_file, "w", encoding="utf-8") as f:
        json.dump(credentials, f, indent=2, ensure_ascii=False)

print("\n" + "-" * 72)
print("Summary")
print("-" * 72)
if do_passwords:
    print(f"Passwords rotated: {stats['password_success']}")
    print(f"Password errors:   {stats['password_error']}")
if do_mfa:
    print(f"MFA reset:         {stats['mfa_success']}")
    print(f"MFA errors:        {stats['mfa_error']}")

if stats["password_error"] > 0 or stats["mfa_error"] > 0:
    raise SystemExit("Some operations failed. See output above.")

print("Done.")

MANAGE USERS (Notebook / SP context)
Action: reset-all
Users: 25
Credentials file: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform\user_credentials.json
WhatIf: False
Active AZ identity: 8a9f736e-4eb2-4484-ae90-2493f57102b3

[user00] user00@cptazure.org
  Password rotated
  Found 1 MFA method(s)
  Removed: microsoftAuthenticatorAuthenticationMethod

[user01] user01@cptazure.org
  Password rotated
  No MFA methods registered

[user02] user02@cptazure.org
  Password rotated
  No MFA methods registered

[user03] user03@cptazure.org
  Password rotated
  No MFA methods registered

[user04] user04@cptazure.org
  Password rotated
  Found 1 MFA method(s)
  Removed: microsoftAuthenticatorAuthenticationMethod

[user05] user05@cptazure.org
  Password rotated
  Found 1 MFA method(s)
  Removed: microsoftAuthenticatorAuthenticationMethod

[user06] user06@cptazure.org
  Password rotated
  Found 2 MFA method(s)
  Removed: microsoftAuthentic

## 3. Get User00 VM Information

Hole die VM-Informationen für User00 aus den Terraform Outputs oder Azure.

In [3]:
# ============================================================================
# Get User00 VM Information
# ============================================================================

USER_INDEX = "00"
VM_NAME = f"vm-user{USER_INDEX}"
RG_NAME = f"rg-vm-user{USER_INDEX}"

# Get VM details from Azure
print(f"🔍 Suche VM: {VM_NAME} in RG: {RG_NAME}")

vm_info = az_json(f'az vm show -g {RG_NAME} -n {VM_NAME} -o json --only-show-errors')
if not vm_info:
    raise SystemExit(f"❌ VM {VM_NAME} nicht gefunden. Stelle sicher dass 'terraform apply' ausgeführt wurde.")

# Get public IP
public_ip = run_output(f'az vm show -g {RG_NAME} -n {VM_NAME} -d --query publicIps -o tsv --only-show-errors')
private_ip = run_output(f'az vm show -g {RG_NAME} -n {VM_NAME} -d --query privateIps -o tsv --only-show-errors')

print(f"✅ VM gefunden:")
print(f"   Name: {VM_NAME}")
print(f"   Resource Group: {RG_NAME}")
print(f"   Public IP: {public_ip}")
print(f"   Private IP: {private_ip}")
print(f"   Location: {vm_info.get('location', 'unknown')}")

# Store for later use
VM_PUBLIC_IP = public_ip
VM_PRIVATE_IP = private_ip

🔍 Suche VM: vm-user00 in RG: rg-vm-user00
✅ VM gefunden:
   Name: vm-user00
   Resource Group: rg-vm-user00
   Public IP: 4.233.91.238
   Private IP: 10.0.0.4
   Location: francecentral


## 4. SSH via Azure CLI (Entra ID)

Verbindung zur VM via `az ssh vm` mit dem Service Principal.
Der SP hat `Virtual Machine Administrator Login` Rechte.

In [4]:
# ============================================================================
# SSH Connection via Azure CLI (Entra ID Authentication) - non-interactive
# ============================================================================
# Ziel: SSH schnell testen, ohne dass irgendwas "hängen" kann (Host-Key Prompt etc.).
# ============================================================================

print("=" * 70)
print("SSH Verbindung zur User00 VM")
print("=" * 70)

if "run" not in globals():
    raise SystemExit("❌ Helper-Funktion run() fehlt. Bitte zuerst Zelle 3 ausführen.")
if "VM_PUBLIC_IP" not in globals() or not VM_PUBLIC_IP:
    raise SystemExit("❌ VM_PUBLIC_IP fehlt. Bitte zuerst Zelle 7 ausführen.")

# Check if ssh extension is installed (timeout)
print("\n🔍 Prüfe ob Azure CLI SSH Extension installiert ist...")
try:
    rc = run("az extension show --name ssh --only-show-errors -o none", shell=True, timeout=30)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout bei 'az extension show' (Azure CLI hängt?)")

if rc != 0:
    print("📦 Installiere SSH Extension...")
    run("az extension add --name ssh --only-show-errors", shell=True, timeout=120)

# Non-interactive SSH options to avoid prompts/hangs
SSH_OPTS = "-o BatchMode=yes -o ConnectTimeout=20 -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null"

print(f"\n🔗 Teste SSH Verbindung zu {VM_PUBLIC_IP} (non-interactive, timeout)...")
print("-" * 70)

ssh_test_cmd = f'az ssh vm --ip {VM_PUBLIC_IP} -- {SSH_OPTS} "echo OK && hostname && whoami"'

try:
    rc = run(ssh_test_cmd, shell=True, timeout=90)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout: SSH Test >90s (Port 22 / AADSSHLoginForLinux / NSG prüfen)")

print("-" * 70)
if rc == 0:
    print("✅ SSH Verbindung erfolgreich!")
else:
    print("⚠️  SSH Verbindung fehlgeschlagen. Mögliche Ursachen:")
    print("   - NSG blockiert SSH (Port 22)")
    print("   - AADSSHLoginForLinux Extension nicht installiert/fehlt")
    print("   - SP hat keine 'Virtual Machine Administrator Login' Rolle")
    print("   - Azure CLI ssh-extension Problem")

SSH Verbindung zur User00 VM

🔍 Prüfe ob Azure CLI SSH Extension installiert ist...

🔗 Teste SSH Verbindung zu 4.233.91.238 (non-interactive, timeout)...
----------------------------------------------------------------------
----------------------------------------------------------------------
✅ SSH Verbindung erfolgreich!


OpenSSH_for_Windows_9.5p2, LibreSSL 3.8.2
8a9f736e-4eb2-4484-ae90-2493f57102b3@4.233.91.238: Permission denied (publickey).



## 5. Check Oracle Tools auf der VM

Prüfe ob die Oracle Tools (SQLcl, Instant Client, etc.) auf der VM installiert sind.

In [5]:
# ============================================================================
# Check Oracle Tools on VM
# ============================================================================

print("=" * 70)
print("Oracle Tools Check auf der VM")
print("=" * 70)

check_commands = """
echo "=== Oracle Instant Client ==="
ls -la /opt/oracle/instantclient* 2>/dev/null || echo "Not found"
echo ""
echo "=== SQLcl ==="
/opt/oracle/sqlcl/bin/sql -V 2>/dev/null || echo "Not found"
echo ""
echo "=== Java ==="
java -version 2>&1 | head -1
echo ""
echo "=== Azure CLI ==="
az --version 2>/dev/null | head -1
echo ""
echo "=== OCI CLI ==="
oci --version 2>/dev/null || echo "Not found"
echo ""
echo "=== rwloadsim/connping ==="
which connping 2>/dev/null || echo "Not found"
echo ""
echo "=== Network Tools ==="
which dig traceroute nc tcpdump 2>/dev/null | head -5
"""

rc = run(f'az ssh vm --ip {VM_PUBLIC_IP} -- "{check_commands}"', shell=True)

if rc == 0:
    print("-" * 70)
    print("✅ Oracle Tools Check abgeschlossen")
else:
    print("⚠️  Check fehlgeschlagen")

Oracle Tools Check auf der VM
----------------------------------------------------------------------
✅ Oracle Tools Check abgeschlossen


OpenSSH_for_Windows_9.5p2, LibreSSL 3.8.2
Pseudo-terminal will not be allocated because stdin is not a terminal.
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
@    WARNING: REMOTE HOST IDENTIFICATION HAS CHANGED!     @
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
IT IS POSSIBLE THAT SOMEONE IS DOING SOMETHING NASTY!
Someone could be eavesdropping on you right now (man-in-the-middle attack)!
It is also possible that a host key has just been changed.
The fingerprint for the ED25519 key sent by the remote host is
SHA256:oF299DCij6KM91me5EYxE50CqrjlnC90gkOgORVngiE.
Please contact your system administrator.
Add correct host key in C:\\Users\\chpinoto/.ssh/known_hosts to get rid of this message.
Offending ECDSA key in C:\\Users\\chpinoto/.ssh/known_hosts:125
Host key for 20.111.56.212 has changed and you have requested strict checking.
Host key verification failed.



## 6. Test ODAA VNet Connectivity

Teste die Netzwerkverbindung zum ODAA VNet (192.168.x.x) über das VNet Peering.

In [ ]:
# ============================================================================
# Test ODAA VNet Connectivity
# ============================================================================
# Das VM VNet (10.0.0.0/16) ist mit dem ODAA VNet (192.168.0.0/16) gepeert.
# Die ADB hat typischerweise eine IP im Bereich 192.168.0.x
# ============================================================================

print("=" * 70)
print("ODAA VNet Connectivity Test")
print("=" * 70)

# Get ODAA VNet information
ODAA_RG = f"rg-odaa-user{USER_INDEX}"
ODAA_VNET = f"vnet-odaa-user{USER_INDEX}"

print(f"\n🔍 Hole ODAA VNet Informationen...")
print(f"   Resource Group: {ODAA_RG}")
print(f"   VNet: {ODAA_VNET}")

# Switch to ODAA subscription temporarily
print(f"\n🔄 Wechsle zu ODAA Subscription...")
run(f'az account set -s {ODAA_SUBSCRIPTION_ID} --only-show-errors', shell=True)

# Get ODAA subnet info
odaa_vnet_info = az_json(f'az network vnet show -g {ODAA_RG} -n {ODAA_VNET} -o json --only-show-errors')
if odaa_vnet_info:
    address_space = odaa_vnet_info.get('addressSpace', {}).get('addressPrefixes', [])
    subnets = odaa_vnet_info.get('subnets', [])
    print(f"✅ ODAA VNet gefunden:")
    print(f"   Address Space: {address_space}")
    for subnet in subnets:
        print(f"   Subnet: {subnet.get('name')} - {subnet.get('addressPrefix')}")
else:
    print(f"⚠️  ODAA VNet {ODAA_VNET} nicht gefunden")

# Switch back to VM subscription
run(f'az account set -s {VM_SUBSCRIPTION_ID} --only-show-errors', shell=True)

# Test connectivity from VM to ODAA subnet
print(f"\n🔗 Teste Netzwerk-Konnektivität von VM zu ODAA VNet...")
print("-" * 70)

connectivity_test = """
echo "=== Network Interfaces ==="
ip addr show | grep -E "inet |^[0-9]"
echo ""
echo "=== Route Table ==="
ip route
echo ""
echo "=== DNS Resolution Test ==="
# Try to resolve Oracle DNS zone
nslookup adb.eu-paris-1.oraclecloud.com 2>/dev/null || echo "DNS lookup failed"
echo ""
echo "=== Ping Test to ODAA Subnet (192.168.0.0/24) ==="
# Ping gateway of ODAA subnet
ping -c 3 -W 2 192.168.0.1 2>/dev/null || echo "Ping to 192.168.0.1 failed (may be blocked by NSG)"
"""

rc = run(f'az ssh vm --ip {VM_PUBLIC_IP} -- "{connectivity_test}"', shell=True)
print("-" * 70)
print("✅ Connectivity Test abgeschlossen")

## Clean Up ODAA